In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = (
    SparkSession.builder
    .appName("BronzeToSilver")
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262")
    .config("spark.hadoop.fs.s3a.endpoint", "http://localhost:9000")
    .config("spark.hadoop.fs.s3a.access.key", "admin")
    .config("spark.hadoop.fs.s3a.secret.key", "password")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .getOrCreate()
)

In [ ]:
# Ler todos os arquivos da camada Bronze no MinIO
df = spark.read.json("s3a://bronze/")
df.printSchema()
df.show(truncate=False)

In [3]:
from pyspark.sql.types import *
from pyspark.sql.functions import *


In [4]:
df = df.withColumn("timestamp", to_timestamp(col("timestamp")))
df

DataFrame[equipment_id: string, event_id: string, factory_id: string, is_anomaly: boolean, measurement_type: string, metadata: struct<battery_level:bigint,firmware_version:string,signal_strength:bigint>, quality: string, sensor_id: string, timestamp: timestamp, unit: string, value: double]

In [5]:
df = df.withColumn("sensor_id", trim(col("sensor_id"))) \
       .withColumn("equipment_id", trim(col("equipment_id"))) \
       .withColumn("factory_id", trim(col("factory_id"))) \
       .withColumn("measurement_type", lower(trim(col("measurement_type")))) \
       .withColumn("unit", lower(trim(col("unit")))) \
       .withColumn("quality", lower(trim(col("quality"))))

In [7]:
df = df.withColumn("firmware_version", col("metadata.firmware_version")) \
       .withColumn("battery_level", col("metadata.battery_level")) \
       .withColumn("signal_strength", col("metadata.signal_strength")) \
       .drop("metadata")

In [8]:
df = df.filter(
    col("event_id").isNotNull() &
    col("sensor_id").isNotNull() &
    col("value").isNotNull() &
    col("timestamp").isNotNull()
)

df = df.dropDuplicates(["event_id"])

In [ ]:
df_final = (
    df
    .withColumn("year", year("timestamp"))
    .withColumn("month", month("timestamp"))
    .withColumn("day", dayofmonth("timestamp"))
)

df_final.write \
    .mode("overwrite") \
    .partitionBy("year", "month", "factory_id") \
    .json("s3a://silver/")

print("Camada Silver salva no MinIO: s3a://silver/")